In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-colorblind')
plt.rcParams['axes.facecolor'] = 'white'

from  sklearn.neighbors import KNeighborsClassifier

In [2]:
# Import distance matrices
dist_dowker = np.loadtxt('../../simHippo/diagrams/distances_dowker_purge.csv',
          delimiter = ',')
dist_walklen = np.loadtxt('../../simHippo/diagrams/distances_walklen_purge.csv',
          delimiter = ',')

# Number of holes labels
labels = [int(i/20) for i in range(100)]

dist_dowker

array([[0.        ,        inf, 0.05063796, ...,        inf,        inf,
               inf],
       [       inf, 0.        ,        inf, ...,        inf,        inf,
               inf],
       [0.05063796,        inf, 0.        , ...,        inf,        inf,
               inf],
       ...,
       [       inf,        inf,        inf, ..., 0.        , 0.02262762,
        0.03388097],
       [       inf,        inf,        inf, ..., 0.02262762, 0.        ,
        0.04058344],
       [       inf,        inf,        inf, ..., 0.03388097, 0.04058344,
        0.        ]])

In [3]:
knn = KNeighborsClassifier(n_neighbors = 5, metric='precomputed')

test_indices = [19,39,59,79,9]
train_walklen = np.delete(np.delete(dist_walklen, test_indices, axis=0), test_indices, axis=1)
train_labels = np.delete(labels, test_indices, axis=0)

knn.fit(train_walklen, train_labels)

KNeighborsClassifier(metric='precomputed')

In [4]:
test_walklen = np.delete(dist_walklen[test_indices,:], test_indices, axis=1)

knn.predict(test_walklen)

array([0, 1, 2, 3, 0])

In [5]:
np.max(dist_dowker)

inf

In [6]:
dist_dowker[dist_dowker == np.inf] = 1

In [7]:
# K-fold Knn

dist_matrix = np.loadtxt('../../simHippo/diagrams/distances_walklen_purge.csv',
          delimiter = ',')

confusion = np.zeros((5,5))

knn = KNeighborsClassifier(n_neighbors = 4, metric='precomputed')
print('----- 4 nearest neighbors -----')

total_num = 100
k = 100
splitting = list(range(total_num))
import random
random.seed(12)
#random.shuffle(splitting)

accuracy = 0
for j in range(k):
    # Take j-th subset as test set
    test_indices = splitting[j*total_num//k:(j+1)*total_num//k]
    test_dist = np.delete(dist_matrix[test_indices,:], test_indices, axis=1)

    # Delete test data from distance matrix and labels list
    train_dist = np.delete(np.delete(dist_matrix, test_indices, axis=0), test_indices, axis=1)
    train_labels = np.delete(labels, test_indices, axis=0)

    knn.fit(train_dist, train_labels)
    pred_labels = knn.predict(test_dist)

    real_labels = np.array(test_indices)//20
    #print(pred_labels[0],real_labels[0])
    confusion[pred_labels[0],real_labels[0]] += 1

    correct_percent = sum(real_labels == pred_labels)*k/total_num
    accuracy += correct_percent
    #print('------')

accuracy /= k
print(accuracy)

----- 4 nearest neighbors -----
0.86


In [8]:
confusion

array([[20.,  6.,  0.,  0.,  0.],
       [ 0., 13.,  2.,  0.,  0.],
       [ 0.,  1., 18.,  4.,  0.],
       [ 0.,  0.,  0., 16.,  1.],
       [ 0.,  0.,  0.,  0., 19.]])

In [7]:
# For all
num_neighbors = 4

knn = KNeighborsClassifier(n_neighbors = num_neighbors, metric='precomputed')
print('----- '+str(num_neighbors)+' nearest neighbors -----')

total_num = 100
k = 100
splitting = list(range(total_num))
import random
random.seed(12345)
random.shuffle(splitting)

# Number of holes labels
labels = [int(i/20) for i in range(100)]

methods = ['ripsmax', 'ripsmin', 'dowker', 'walklen']

for method in methods:
    if method == 'dowker':
        preprocs = ['original', 'min1', 'minpurge', 'purge','originalshort','min1short', 'minpurgeshort', 'purgeshort']
        #preprocs = ['original', 'min1', 'originalshort', 'min1short', 'minpurgeshort', 'purgeshort']
    elif method == 'walklen' or method == 'ripsmax' or method == 'ripsmin':
        preprocs = ['original', 'min1', 'minpurge', 'purge']
    
    for preproc in preprocs:
        dist_matrix = np.loadtxt('../../simHippo/diagrams/distances_'+method+'_'+preproc+'.csv',
                                 delimiter = ',')

        confusion = np.zeros((5,5))

        dist_matrix[ dist_matrix == np.inf ] = 100

        accuracy = 0
        for j in range(k):
            # Take j-th subset as test set
            test_indices = splitting[j*total_num//k:(j+1)*total_num//k]
            test_dist = np.delete(dist_matrix[test_indices,:], test_indices, axis=1)

            # Delete test data from distance matrix and labels list
            train_dist = np.delete(np.delete(dist_matrix, test_indices, axis=0), test_indices, axis=1)
            train_labels = np.delete(labels, test_indices, axis=0)

            knn.fit(train_dist, train_labels)
            pred_labels = knn.predict(test_dist)
        
            real_labels = np.array(test_indices)//20
            confusion[pred_labels[0],real_labels[0]] += 1
            #print(real_labels)
            #print(pred_labels)
        
            correct_percent = sum(real_labels == pred_labels)*k/total_num
            accuracy += correct_percent

        accuracy /= k
        print(int(accuracy*100), method, preproc)

        ## Confusion matrix

        plt.figure(figsize=(5,5))
        plt.gca().yaxis.set_ticks_position('right')
        plt.gca().yaxis.set_label_position('right')
        plt.imshow(confusion, cmap='Greens', vmin=0, vmax=20)

        for i in range(5):
            for j in range(5):
                c = 'w'
                if confusion[i, j]<10 and confusion[i, j]!=0: c = 'g'
                plt.text(j, i, int(confusion[i, j]), ha="center", va="center", color=c, size=10.5, weight='bold')
        plt.grid(False)
        plt.xlabel('Expected')
        plt.ylabel('Predicted')
        plt.suptitle(method+'_'+preproc)
        #plt.savefig('../figures/confusion_'+method+'_'+preproc+'.png', bbox_inches='tight', dpi=400)
        #plt.show()
        plt.close()
    print('***')

----- 4 nearest neighbors -----
57 ripsmax original
51 ripsmax min1
51 ripsmax minpurge
59 ripsmax purge
***
37 ripsmin original
56 ripsmin min1
63 ripsmin minpurge
73 ripsmin purge
***
50 dowker original
71 dowker min1
73 dowker minpurge
82 dowker purge
49 dowker originalshort
71 dowker min1short
71 dowker minpurgeshort
82 dowker purgeshort
***
7 walklen original
80 walklen min1
76 walklen minpurge
86 walklen purge
***
